In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

np.random.seed(42)
tf.random.set_seed(42)

vocab_size = 15000  
max_len = 250       

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

x_train_padded = pad_sequences(x_train, maxlen=max_len, padding='pre', truncating='pre')
x_test_padded = pad_sequences(x_test, maxlen=max_len, padding='pre', truncating='pre')

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)
]

history = model.fit(
    x_train_padded,
    y_train,
    epochs=4,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_accuracy = model.evaluate(x_test_padded, y_test, verbose=0)
error_rate = 1.0 - test_accuracy

print("\n" + "="*50)
print(f"Итоговая точность на тесте (Accuracy): {test_accuracy * 100:.2f}%  (Требуется: >= 85%)")
print(f"Количество ошибок (Error Rate):        {error_rate * 100:.2f}%   (Требуется: <= 15%)")
print("="*50 + "\n")

y_pred_probs = model.predict(x_test_padded, batch_size=128)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=['Негативный', 'Позитивный']))

17464789/17464789 [==============================] - 3s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 128)         1920000   
                                                                 
 bidirectional (Bidirectiona  (None, None, 128)        98816     
 l)                                                              
                                                                 
 global_max_pooling1d (Globa  (None, 128)              0         
 lMaxPooling1D)                                                  
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                         

In [3]:
word_index = imdb.get_word_index()
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_seq):
    return ' '.join([reverse_word_index.get(i, '?') for i in encoded_seq if i > 3])

sample_size = 10
sample_preds = model.predict(x_test_padded[:sample_size], verbose=0)
sample_labels = (sample_preds > 0.5).astype(int).flatten()

correct_count = 0

for i in range(sample_size):
    text = decode_review(x_test[i])
    real_label = y_test[i]
    pred_label = sample_labels[i]
    prob = sample_preds[i][0]
    
    is_correct = (real_label == pred_label)
    if is_correct:
        correct_count += 1
        
    print(f"\nПример {i + 1}:")
    print(f"Текст отзыва: {text[:180]}...")
    print(f"Реальная метка:             {real_label} ({'Позитив' if real_label == 1 else 'Негатив'})")
    print(f"Предсказанная вероятность:  {prob:.4f}")
    print(f"Предсказанная метка:        {pred_label} ({'Позитив' if pred_label == 1 else 'Негатив'})")
    print(f"Совпадение:                 {'Да' if is_correct else 'Нет'}")

print(f"\nИТОГО из {sample_size} примеров совпало: {correct_count} ({correct_count/sample_size * 100:.0f}%)")

1641221/1641221 [==============================] - 3s 2us/step

Пример 1:
Текст отзыва: please give this one a miss br br swanson and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allow...
Реальная метка:             0 (Негатив)
Предсказанная вероятность:  0.0373
Предсказанная метка:        0 (Негатив)
Совпадение:                 Да

Пример 2:
Текст отзыва: this film requires a lot of patience because it focuses on mood and character development the plot is very simple and many of the scenes take place on the same set in frances auste...
Реальная метка:             1 (Позитив)
Предсказанная вероятность:  0.9921
Предсказанная метка:        1 (Позитив)
Совпадение:                 Да

Пример 3:
Текст отзыва: many animation buffs consider the great forgotten genius of one special branch of the art puppet animation which he invented almost single handedly and as it happened almost accide...
Реальная метка:   